# IR 0 — Acquire NIST WebBook IR data and write reloadable Parquet files

This pre-work notebook systematically queries the [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/) through NISTChemPy, using the retained list of NIST compound identifiers. For each successful record it retrieves the first IR JCAMP spectrum, a 2-D molfile, canonical SMILES, metadata, and reproducible RDKit SMARTS functional-group labels.

Two Parquet files make acquisition resumable and inspectable. `outputs/nist_acquisition/ir_raw_checkpoint.parquet` contains the downloaded, native `x_raw`/`y_raw` arrays and can be reloaded after an interruption. `IR_SMILES_functional_groups.parquet` contains finite, normalized spectra on one common 500-point wavenumber grid for IR 1. Start with a small pilot; use `MAX_COMPOUNDS = None` only for a deliberate, resumable full collection.


In [1]:
import sys
from importlib.util import find_spec
from io import StringIO
from pathlib import Path

if find_spec('nistchempy') is None:
    raise ModuleNotFoundError(
        'NISTChemPy is missing from this notebook kernel. Select the workshop .venv kernel, then run: uv sync --locked'
    )

import jcamp
import nistchempy as nist
import numpy as np
import pandas as pd
from rdkit import Chem
from tqdm.auto import tqdm

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs/nist_acquisition')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ID_FILE = DATA_DIR / 'compounds.parquet'
RAW_CHECKPOINT = OUTPUT_DIR / 'ir_raw_checkpoint.parquet'
FAILURE_LOG = OUTPUT_DIR / 'retrieval_failures.parquet'
OUTPUT_FILE = OUTPUT_DIR / 'IR_SMILES_functional_groups.parquet'

RUN_LIVE_COLLECTION = True   # Set False to inspect/rebuild an existing checkpoint without network access.
MAX_COMPOUNDS = 10           # Pilot batch. Set None for a full, resumable crawl after a successful pilot.
RETRY_PREVIOUS_FAILURES = False  # Keep False to advance through IDs without retrying known no-spectrum records.
CHECKPOINT_EVERY = 25
TARGET_POINTS = 500
REBUILD_STANDARDIZED_TABLE = True
request_config = nist.RequestConfig(delay=3.0, max_attempts=3, kwargs={'timeout': 30.0})

if nist.__version__ != '2.0.0':
    raise RuntimeError(f'Tested with nistchempy 2.0.0; found {nist.__version__}.')
compound_ids = pd.read_parquet(ID_FILE)['data'].dropna().astype(str).drop_duplicates().tolist()
print(f'NistChemPy {nist.__version__}; {len(compound_ids):,} candidate IDs; live collection={RUN_LIVE_COLLECTION}')


NistChemPy 2.0.0; 68,030 candidate IDs; live collection=True


In [2]:
SMARTS = {
    'Vinylic': '[$([CX3]=[CX3])]', 'Arene': 'c', 'Alcohol': '[#6][OX2H]',
    'Ether': '[OD2]([#6])[#6]', 'Amine': '[NX3;H2,H1;!$(NC=O)]', 'Nitrile': '[NX1]#[CX2]',
    'Aldehyde': '[CX3H1](=O)[#6]', 'Ketone': '[#6][CX3](=O)[#6]',
    'CarboxylAcid': '[OX2H][CX3]=[OX1]', 'Ester': '[#6][CX3](=O)[OX2H0][#6]',
    'Amide': '[NX3][CX3](=[OX1])[#6]', 'AcidChloride': '[CX3](=[OX1])[F,Cl,Br,I]',
}
PATTERNS = {name: Chem.MolFromSmarts(pattern) for name, pattern in SMARTS.items()}

def labels_from_smiles(smiles):
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError('NIST 2-D molfile produced invalid SMILES.')
    return {name: int(molecule.HasSubstructMatch(pattern)) for name, pattern in PATTERNS.items()}

def fetch_record(compound_id):
    compound = nist.get_compound(compound_id, request_config=request_config)
    if compound is None:
        raise LookupError('No unique NIST record.')
    spectra = compound.get_ir_spectra() or []
    if not spectra:
        raise LookupError('No IR spectrum.')
    parsed = jcamp.jcamp_read(StringIO(spectra[0].jdx_text))
    x = np.asarray(parsed.get('x', []), dtype=float)
    y = np.asarray(parsed.get('y', []), dtype=float)
    if len(x) < 2 or len(x) != len(y) or not (np.isfinite(x).all() and np.isfinite(y).all()):
        raise ValueError('Invalid JCAMP x/y arrays.')
    molecule = Chem.MolFromMolBlock(compound.get_mol2D(), sanitize=True, removeHs=True)
    if molecule is None:
        raise ValueError('NIST 2-D molfile cannot be parsed.')
    smiles = Chem.MolToSmiles(molecule, canonical=True)
    record = {
        'NIST_molecule': compound_id, 'CAS_NO': compound.cas_rn,
        'Name': parsed.get('title') or compound.name, 'molform': parsed.get('molform') or compound.formula,
        'SMILES': smiles, 'x_raw': x.tolist(), 'y_raw': y.tolist(),
        'xunits': str(parsed.get('xunits', '')).upper(), 'yunits': str(parsed.get('yunits', '')).upper(),
    }
    record.update(labels_from_smiles(smiles))
    return record


In [3]:
if RUN_LIVE_COLLECTION:
    records = pd.read_parquet(RAW_CHECKPOINT).to_dict('records') if RAW_CHECKPOINT.exists() else []
    previous_failures = pd.read_parquet(FAILURE_LOG).to_dict('records') if FAILURE_LOG.exists() else []
    completed = {record['NIST_molecule'] for record in records}
    if not RETRY_PREVIOUS_FAILURES:
        completed.update(record['NIST_molecule'] for record in previous_failures)
    candidates = [identifier for identifier in compound_ids if identifier not in completed]
    if MAX_COMPOUNDS is not None:
        candidates = candidates[:MAX_COMPOUNDS]
    failures = []
    for count, identifier in enumerate(tqdm(candidates, desc='Downloading NIST WebBook IR records'), start=1):
        try:
            records.append(fetch_record(identifier))
        except Exception as error:
            failures.append({'NIST_molecule': identifier, 'error': repr(error)})
        if count % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_parquet(RAW_CHECKPOINT, index=False)
    pd.DataFrame(records).to_parquet(RAW_CHECKPOINT, index=False)
    all_failures = pd.DataFrame([*previous_failures, *failures])
    if all_failures.empty:
        all_failures = pd.DataFrame(columns=['NIST_molecule', 'error'])
    else:
        all_failures = all_failures.drop_duplicates('NIST_molecule', keep='last')
    all_failures.to_parquet(FAILURE_LOG, index=False)
    all_failures.to_csv(OUTPUT_DIR / 'retrieval_failures.csv', index=False)
    print(f'Raw checkpoint: {len(records):,} records; failures this run: {len(failures):,}; retained failures: {len(all_failures):,}.')
else:
    print('Live collection is disabled. Existing raw Parquet, if present, can still be reloaded below.')


Raw checkpoint: 2 records; failures this run: 10; retained failures: 10.


In [4]:
def standardize_to_model_table(records, n_points=TARGET_POINTS):
    frame = pd.DataFrame(records).dropna(subset=['CAS_NO', 'SMILES']).drop_duplicates('CAS_NO').copy()
    keep = frame.apply(lambda row: len(row.x_raw) == len(row.y_raw) and len(row.x_raw) > 1
                       and row.xunits != 'MICROMETERS' and row.yunits != 'TRANSMITTANCE', axis=1)
    spectra = []
    for row in frame.loc[keep].to_dict('records'):
        x, y = np.asarray(row['x_raw'], dtype=float), np.asarray(row['y_raw'], dtype=float)
        order = np.argsort(x); x, y = x[order], y[order]
        x, inverse = np.unique(x, return_inverse=True)
        y = np.bincount(inverse, weights=y) / np.bincount(inverse)
        if len(x) > 1 and np.isfinite(x).all() and np.isfinite(y).all():
            spectra.append((row, x, y))
    if not spectra:
        raise ValueError('No compatible finite raw spectra remain to standardize.')
    low, high = max(x.min() for _, x, _ in spectra), min(x.max() for _, x, _ in spectra)
    if not low < high:
        raise ValueError('No common wavenumber interval remains.')
    grid, rows = np.linspace(low, high, n_points), []
    for row, x, y in spectra:
        values = np.interp(grid, x, y)
        span = values.max() - values.min()
        if span > 0:
            output = {key: row[key] for key in ['NIST_molecule', 'CAS_NO', 'Name', 'molform', 'SMILES'] + list(SMARTS)}
            output['wavenumber'] = grid.tolist()
            output['IR'] = ((values - values.min()) / span).tolist()
            rows.append(output)
    return pd.DataFrame(rows)

if RAW_CHECKPOINT.exists():
    raw_records = pd.read_parquet(RAW_CHECKPOINT)
    print(f'Reloaded raw checkpoint: {len(raw_records):,} rows from {RAW_CHECKPOINT}')
    display(raw_records[['NIST_molecule', 'CAS_NO', 'Name', 'xunits', 'yunits']].head())
    if REBUILD_STANDARDIZED_TABLE:
        model_table = standardize_to_model_table(raw_records.to_dict('records'))
        model_table.to_parquet(OUTPUT_FILE, index=False)
        print(f'Wrote {len(model_table):,} standardized labeled spectra to {OUTPUT_FILE}.')
else:
    print(f'No raw checkpoint yet. Enable RUN_LIVE_COLLECTION to create {RAW_CHECKPOINT}.')


Reloaded raw checkpoint: 2 rows from outputs/nist_acquisition/ir_raw_checkpoint.parquet


,NIST_molecule,CAS_NO,Name,xunits,yunits
0,C106365,106-36-5,"Propanoic acid, propyl ester",1/CM,ABSORBANCE
1,C91342335,91342-33-5,"CYCLOHEXYLAMINE, N-SEC-BUTYL-, HYDROCHLORIDE",MICROMETERS,TRANSMITTANCE


Wrote 1 standardized labeled spectra to outputs/nist_acquisition/IR_SMILES_functional_groups.parquet.


In [5]:
if OUTPUT_FILE.exists():
    fresh = pd.read_parquet(OUTPUT_FILE)
    required = {'IR', 'wavenumber', 'SMILES'} | set(SMARTS)
    if not required.issubset(fresh.columns) or not fresh.IR.map(len).eq(TARGET_POINTS).all():
        raise ValueError('The standardized Parquet file does not satisfy the IR 1 input contract.')
    if not all(np.array_equal(np.asarray(fresh.wavenumber.iloc[0]), np.asarray(values)) for values in fresh.wavenumber):
        raise ValueError('Standardized spectra do not share one wavenumber grid.')
    print(f'Validated reloadable IR 1 input: {len(fresh):,} rows, {TARGET_POINTS} points, {len(SMARTS)} binary labels.')
else:
    print('No standardized output yet; IR 1 will use its packaged rehearsal fallback.')


Validated reloadable IR 1 input: 1 rows, 500 points, 12 binary labels.
